# Translation quality

Score aligned Ukrainian and English triplets with reference-free COMET. Save one
score for each sentence field and review the lowest scores. Scores do not mark
human annotation complete.

Install `.[quality,notebooks]`. Set `RUN_SCORING = True` to fetch the fixed
reference and run COMET. Use a new output directory for each run.


In [ ]:
import json
from pathlib import Path

from fairforget.io import file_digest, read_rows, write_json, write_rows
from fairforget.translation.quality import load_comet_scorer, prepare_quality, score_quality
from fairforget.translation.reference import load_reference, validate_reference

project = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base / "warbias", base)
    if (candidate / "configs/translation/quality.json").is_file()
)
model_config = json.loads((project / "configs/translation/quality.json").read_text())
reference_config = validate_reference(
    json.loads((project / "configs/data/warbias-reference.json").read_text())
)

RUN_SCORING = False
INPUT_KIND = "reference"
UKRAINIAN_PATH = Path("data/uk.jsonl")
ENGLISH_PATH = Path("data/en.jsonl")
OUTPUT = Path("runs/translation-quality")

model_config

The default data are a pinned published reference. Its authorship includes older
English-first and bilingual edits; this check does not assign it production
Ukrainian-to-English lineage.

For current production output, set `INPUT_KIND = "production"` and the two input
paths above. That mode requires the English rows to match their Ukrainian
source hashes. Human annotation can remain in progress.

The default runtime uses CPU. To use Apple MPS, set `gpus` to `1` and `accelerator`
to `"mps"` in `model_config["prediction_options"]`.


In [ ]:
if INPUT_KIND not in {"reference", "production"}:
    raise ValueError("Choose reference or production input")

if RUN_SCORING:
    if INPUT_KIND == "reference":
        ukrainian, english, origin = load_reference(reference_config)
    else:
        ukrainian, english = read_rows(UKRAINIAN_PATH), read_rows(ENGLISH_PATH)
        origin = {
            "kind": "production_translation",
            "files_sha256": {
                "uk": file_digest(UKRAINIAN_PATH),
                "en": file_digest(ENGLISH_PATH),
            },
        }
    plan = prepare_quality(
        ukrainian,
        english,
        input_origin=origin,
        verify_translation_bindings=INPUT_KIND == "production",
    )
    OUTPUT.mkdir(parents=True, exist_ok=False)
    write_json(OUTPUT / "quality-plan.json", plan)
    scorer, identity = load_comet_scorer(
        model_config["model_id"], revision=model_config["revision"]
    )
    report = score_quality(
        plan,
        scorer=scorer,
        model_identity=identity,
        prediction_options=model_config["prediction_options"],
    )
    write_json(OUTPUT / "quality-result.json", report)
    write_rows(OUTPUT / "segment-scores.jsonl", report["segments"])
    result = {
        "status": "scored",
        "input_kind": INPUT_KIND,
        "coverage": report["coverage"],
        "summary": report["summary"],
        "by_field": report["by_field"],
    }
else:
    result = {"status": "not_run", "input_kind": INPUT_KIND}
result

COMET QE20 returns raw learned scores, including possible negative values. There
is no acceptance cutoff. The table below selects 20 sentences for review; it does
not reject or accept them.


In [ ]:
if RUN_SCORING:
    from IPython.display import display
    from pandas import DataFrame

    review = DataFrame(sorted(report["segments"], key=lambda row: row["score"])[:20])
    display(review[["id", "field", "score", "src", "mt"]])